# MTPFN baseline: hierarchical multi-task PFN, reproduced on its robust ICM prior and run on the AIR A→B prior

## Description

*Written 2026-11-03, before the first tracked run. Issue #20, branch `exp/20-mtpfn`.*

**Paper.** Y. L. Li, S. Daulton, S. Müller, A. G. Wilson, E. Bakshy, *Robust Transfer for Bayesian Optimization with Prior-Data Fitted Networks*, accepted at the 3rd SPIGM workshop @ NeurIPS 2025. The same paper is also listed under the title *Robust Multi-task Modeling for Bayesian Optimization via In-Context Learning*. No arXiv id or code was found (`status: to-verify`). Both PDFs were read in full, including appendices A–E.

**Question.** MTPFN is the published PFN for in-context transfer from source tasks to a target task, so it is the natural external baseline for our A→B question.
- **(R) Reproduction.** Does our implementation reproduce the paper's claims on the paper's own prior?
  - It predicts close to the Bayes predictive (§4.3, Fig. A.5).
  - It transfers from related sources.
  - Training with unrelated sources (p > 0) makes it robust to them (Fig. 3, Fig. A.1).
- **(A) Baseline.** Trained on our AIR prior (B a dense, coordinate-warped source; A a scarce target), how does it compare with AIR (pilot 3) and with the clean A-only PFN, on identical evaluation tasks?

**Model as built** (`experimental/mtpfn/model.py`, from pfn-repro's `PFNBlock` / `MaskedMHA`, `BarDistribution`, `_pad_and_rescale_features`, `cosine_warmup_lr`):
- **Inputs.** A [TASK] token per task, plus point tokens `Linear(x, y)`. The target is task 0, and its [TASK] token carries a learned target flag.
- **Blocks.** Feature encoder → N × [intra-task block, inter-task block] → intra-task block → output layer.
  - Intra-task block: a task's [TASK] and point tokens attend bidirectionally to each other. The target's query tokens cross-attend to the target's tokens only.
  - Inter-task block: the T [TASK] tokens attend to each other. **This is the only route between tasks.**
- **Training.** AdamW, lr 1e-4, cosine annealing (as in the paper), NLL of the observed y at the target's queries, gradient clip 1.

**Prior R** (`icm.py`, Alg. A.1):
- x ~ U[0,1]^d; Dirichlet(1) task proportions; K_T ~ LKJ(1); ℓ ~ Gamma(3, rate 6); y ~ N(0, K_T ⊗ K_X).
- Each source is unrelated with probability p: an independent RBF GP with its own ℓ_j.

**Deviations from the paper** (forced by missing details or compute):
1. **Size.** 9 blocks, width 256, 4 heads, d_ff 512 (paper: 23 blocks, width 512).
2. **Budget.** 100k steps × batch 64 = 6.4M datasets (paper: about 50M, batch 16).
3. **Noise.** The noise prior is not given; we use σ ~ U[0.01, 0.1] per dataset.
4. **Design ranges.** T ∈ {2..5}, d ∈ {1..3}, n_tot ∈ [10, 100] points per dataset; the target always gets ≥ 1 point.
5. **Output head.** The bar distribution, 400 bins on [−5, 5], is the main head (pfn-repro standard). Fig. 2 labels the output (μ, σ²), so a Gaussian head is trained as an ablation.
6. **Target marking.** The paper does not say how the target is marked; we use a flag.

**Setup.**

*Arm R* (one seed; a reproduction check, not a headline):
- **Training.** MTPFN at p = 0 (bar), p = 0.2 (bar) and p = 0.2 (Gaussian).
- **Evaluation design** (as Fig. A.1: 3 sources × 20 points):
  - T = 4; d ∈ {1, 3}; n_target ∈ {1, 2, 5, 10}.
  - u ∈ {0, 1, 2, 3} sources forced unrelated.
  - 128 datasets per cell, 50 target queries each.
- **References** on the same datasets (exact GP algebra, float64):
  - `bayes`: the Bayes predictive under p = 0.2. Prior importance sampling over (K_T, ℓ, σ, ℓ_j), 1024 draws, with all 2³ unrelated-masks enumerated exactly.
  - `bayes_p0`: the same under p = 0.
  - `ml_icm`: the draw with the best marginal likelihood, a stand-in for the paper's MAP ICM.
  - `oracle_icm`: the true hyperparameters.
  - `gp_target_bayes` / `gp_target_oracle`: target data only.

*Arm A* (3 seeds):
- **Training.** The same architecture at width 128 (1.84M parameters, AIR's decoder + proposal budget) on the pilot-3 prior: ECDF g, H0, Z0, J ∈ {2, 4, 8, 16}, N_B = 128, n_A ~ U{1..10}. 100k steps × 64.
- **Evaluation.** Pilot 3's evaluation tasks (`eval_tasks(J, 256, 777, ecdf)`), J ∈ {2, 16}, n_A ∈ {1, 2, 3, 5, 10}, same bar support. Rows join pilot 3's `nll` table:
  - `a_only_clean`;
  - `isref`;
  - `grid` (J = 2);
  - `qis{K}` / `qmc{K}` at every Σ.

**Pre-registered checks** (mean NLL per query; R1–R3 at d ∈ {1, 3} separately; reference cells with median ESS < 20 are reported but excluded):

| ID | Claim | Pass |
|---|---|---|
| R1 | Bayes fidelity (§4.3) | NLL(MTPFN p=0.2) − NLL(`bayes`) ≤ 0.05 at d = 1 and ≤ 0.10 at d = 3, averaged over n_target and u |
| R2 | Robustness to unrelated sources (Fig. 3/A.1) | at u = 3: NLL(MTPFN p=0) − NLL(MTPFN p=0.2) ≥ 0.05, and NLL(MTPFN p=0.2) ≤ NLL(`gp_target_bayes`) + 0.05 |
| R3 | Transfer | at u = 0, n_target ≤ 5: NLL(`gp_target_bayes`) − NLL(MTPFN p=0.2) ≥ 0.10 |
| R4 | Bayes vs MAP (Fig. A.5) | reported: `bayes`, `ml_icm`, MTPFN by n_target |
| R5 | Gaussian vs bar head | reported |
| A1 | Transfer gain over the clean A-only PFN | reported per (J, n_A): mean over 3 seeds ± SE (paired by task, seed s with pilot-3 seed s) |
| A2 | Gap to the references | reported: MTPFN − `isref`; MTPFN − `grid` (J = 2) |
| A3 | MTPFN vs AIR | "AIR better" if NLL(AIR, best Σ, K = 32) < NLL(MTPFN) − 2 SE at every n_A ≥ 2, for both J. "MTPFN better" is the mirror image. Otherwise "no clear winner". Also reported at K = 8 and K = 128 |

**Expectation, written before the run.**
- Arm R: the MTPFN should come close to `bayes`, more so at d = 1.
- Arm A: MTPFN's tasks talk only through one [TASK] vector per task and layer. That suits ICM-style relations (shared lengthscale, correlation), which are global summaries. It suits a warped dense B badly, because using B there means reading B pointwise at T(x). So we expect a clear gain over A-only at small n_A, but well short of `isref` and of AIR.

**GO / NO-GO.**
- **GO:** R1, R2 and R3 pass. The reproduction is faithful enough, and MTPFN (arm A) becomes the external baseline in the AIR comparison.
- **NO-GO:** any of R1–R3 fails. We investigate (size, budget, head) before using it as a baseline, and arm A's numbers are reported as "under-reproduced".
- **INCONCLUSIVE:** reference ESS < 20 in more than half of the cells.

**Limits.**
- Predictive NLL only. The paper's main results are BO regret on HPOBench / FC-Net; we do not run BO.
- Smaller model and budget than the paper; one seed in arm R.
- Arm A is d = 1, Z0, H0 (pilot 3's scope).
- The arm-R bar support clips |y| > 5. Under unit signal variance plus noise this has probability < 1e-6.

In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/mtpfn/<sweep_id>"  # set after `just snapshot runs/mtpfn/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.